# Task 1: Data Quality Audit

Polars-based audit of `parcel_readings.csv` and `parcel_metadata.csv`.

For each issue we record: **what it is**, **how prevalent**, and **recommended action** (drop / impute / flag / repair).

In [2]:
from pathlib import Path

import polars as pl

ROOT = Path(".")
READINGS_PATH = ROOT / "resources" / "parcel_readings.csv"
META_PATH = ROOT / "resources" / "parcel_metadata.csv"

NDVI_MIN = -1.0
NDVI_MAX = 1.0

READINGS_COLS = [
    "parcel_id",
    "date",
    "ndvi_value",
    "temperature_c",
    "rainfall_mm",
    "sensor_status",
]
META_COLS = [
    "parcel_id",
    "mill_id",
    "crop_type",
    "sowing_date",
    "area_hectares",
]

In [3]:
# Load as Utf8 to preserve raw formatting issues (whitespace, casing, date strings)
readings = pl.read_csv(
    READINGS_PATH,
    schema_overrides={col: pl.Utf8 for col in READINGS_COLS},
)
metadata = pl.read_csv(
    META_PATH,
    schema_overrides={col: pl.Utf8 for col in META_COLS},
)

N_READINGS = readings.height
N_METADATA = metadata.height

print(f"Readings: {N_READINGS:,} rows x {readings.width} cols")
print(readings.schema)
display(readings.head(5))

print(f"\nMetadata: {N_METADATA:,} rows x {metadata.width} cols")
print(metadata.schema)
display(metadata.head(5))

Readings: 3,447 rows x 6 cols
Schema([('parcel_id', String), ('date', String), ('ndvi_value', String), ('temperature_c', String), ('rainfall_mm', String), ('sensor_status', String)])


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
str,str,str,str,str,str
"""PARCEL_018""","""16/05/2026""","""0.595""","""15.4""","""0.0""","""error"""
"""PARCEL_014""","""2026-01-27""","""0.457""","""27.6""","""0.0""","""OK"""
"""PARCEL_006""","""2026-05-17""","""0.497""","""25.8""","""0.0""","""OK"""
"""PARCEL_004""","""10/02/2026""","""0.81""","""25.0""","""0.0""","""OK"""
"""PARCEL_002""","""2026-01-17""","""0.269""","""33.2""","""0.0""","""OK"""



Metadata: 28 rows x 5 cols
Schema([('parcel_id', String), ('mill_id', String), ('crop_type', String), ('sowing_date', String), ('area_hectares', String)])


parcel_id,mill_id,crop_type,sowing_date,area_hectares
str,str,str,str,str
"""PARCEL_001""","""MILL_NORTH""","""sugarcane""","""2026-02-10""","""9.03"""
"""PARCEL_002""","""MILL_SOUTH""","""sugarcane""","""2026-01-16""","""8.97"""
"""PARCEL_003""","""MILL_NORTH""","""soybean""","""2026-02-13""","""5.35"""
"""PARCEL_004""","""MILL_NORTH""","""sugarcane""","""2026-01-02""","""3.18"""
"""PARCEL_005""","""MILL_NORTH""","""soybean""","""2026-02-08""","""2.79"""


In [4]:
def pct(n: int, total: int) -> str:
    if total == 0:
        return "0.0%"
    return f"{100 * n / total:.1f}%"


def has_leading_trailing_ws(df: pl.DataFrame, col: str) -> pl.DataFrame:
    return df.filter(pl.col(col) != pl.col(col).str.strip_chars())


def empty_or_null_mask(col: str) -> pl.Expr:
    stripped = pl.col(col).str.strip_chars()
    return stripped.is_null() | (stripped == "") | stripped.str.to_uppercase().is_in(["NA", "NAN", "NULL", "NONE"])


def date_format_bucket(date_col: str) -> pl.Expr:
    c = pl.col(date_col)
    return (
        pl.when(c.str.contains(r"^\d{4}-\d{2}-\d{2}$"))
        .then(pl.lit("ISO (YYYY-MM-DD)"))
        .when(c.str.contains(r"^\d{2}/\d{2}/\d{4}$"))
        .then(pl.lit("DD/MM/YYYY"))
        .when(c.str.contains(r"^\d{2}-[A-Za-z]{3}-\d{4}$"))
        .then(pl.lit("DD-Mon-YYYY"))
        .otherwise(pl.lit("other"))
        .alias("date_format")
    )


def parse_dates_multi(date_col: str) -> pl.Expr:
    c = pl.col(date_col)
    return (
        pl.coalesce(
            c.str.to_date("%Y-%m-%d", strict=False),
            c.str.to_date("%d/%m/%Y", strict=False),
            c.str.to_date("%d-%b-%Y", strict=False),
        )
        .alias("parsed_date")
    )


def normalize_sensor_status(col: str = "sensor_status") -> pl.Expr:
    stripped = pl.col(col).str.strip_chars().str.to_uppercase()
    return (
        pl.when(stripped.is_in(["", "NA", "NAN", "NULL", "NONE"]))
        .then(pl.lit("MISSING"))
        .when(stripped == "OK")
        .then(pl.lit("OK"))
        .when(stripped == "ERROR")
        .then(pl.lit("ERROR"))
        .otherwise(stripped)
        .alias("sensor_status_normalized")
    )


def build_issue_row(
    dataset: str,
    field: str,
    issue: str,
    count: int,
    total: int,
    action: str,
    justification: str,
) -> dict:
    return {
        "dataset": dataset,
        "field": field,
        "issue": issue,
        "count": count,
        "pct": pct(count, total),
        "action": action,
        "justification": justification,
    }


issue_rows: list[dict] = []

## parcel_readings.csv audit

In [5]:
print("=" * 60)
print("parcel_id")
print("=" * 60)

null_parcel_id = readings.filter(empty_or_null_mask("parcel_id")).height
ws_parcel_id = has_leading_trailing_ws(readings, "parcel_id").height
meta_ids = metadata.select("parcel_id")
orphan_ids = (
    readings.select("parcel_id")
    .unique()
    .join(meta_ids, on="parcel_id", how="anti")
    .sort("parcel_id")
)
orphan_rows = readings.join(orphan_ids, on="parcel_id", how="inner").height

print(f"Null/empty parcel_id: {null_parcel_id} ({pct(null_parcel_id, N_READINGS)})")
print(f"Whitespace in parcel_id: {ws_parcel_id} ({pct(ws_parcel_id, N_READINGS)})")
print(f"Orphan parcel_ids (not in metadata): {orphan_ids.height} parcels, {orphan_rows} rows")
display(orphan_ids)

if orphan_rows:
    issue_rows.append(
        build_issue_row(
            "readings",
            "parcel_id",
            "Orphan parcel_id not in metadata",
            orphan_rows,
            N_READINGS,
            "flag",
            "No metadata to join; exclude from joined output or alert upstream.",
        )
    )

print("\n" + "=" * 60)
print("date")
print("=" * 60)

date_fmt_counts = (
    readings.with_columns(date_format_bucket("date"))
    .group_by("date_format")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
display(date_fmt_counts)

for row in date_fmt_counts.iter_rows(named=True):
    if row["date_format"] != "ISO (YYYY-MM-DD)":
        issue_rows.append(
            build_issue_row(
                "readings",
                "date",
                f"Non-ISO date format: {row['date_format']}",
                row["count"],
                N_READINGS,
                "repair",
                "Parse to canonical Date using explicit format list.",
            )
        )

parsed = readings.with_columns(parse_dates_multi("date"))
unparsed_dates = parsed.filter(pl.col("parsed_date").is_null()).height
print(f"Unparseable dates: {unparsed_dates} ({pct(unparsed_dates, N_READINGS)})")

ambiguous_slash = (
    readings.filter(pl.col("date").str.contains(r"^\d{2}/\d{2}/\d{4}$"))
    .with_columns(
        pl.col("date").str.split("/").list.get(0).cast(pl.Int64).alias("day"),
        pl.col("date").str.split("/").list.get(1).cast(pl.Int64).alias("month"),
    )
    .filter((pl.col("day") <= 12) & (pl.col("month") <= 12))
)
ambig_date_count = ambiguous_slash.select("date").unique().height
ambig_row_count = ambiguous_slash.height
print(f"Ambiguous slash dates (day<=12 and month<=12): {ambig_date_count} unique dates, {ambig_row_count} rows")

if ambig_row_count:
    issue_rows.append(
        build_issue_row(
            "readings",
            "date",
            "Ambiguous DD/MM slash dates (both parts <= 12)",
            ambig_row_count,
            N_READINGS,
            "Repair + Verify",
            "Safe parsing: Infer locale (DD/MM vs MM/DD) from unambiguous rows in the same batch before converting to ISO and set a flag is_data_ambiguous = 1.",
        )
    )

dup_keys = (
    readings.group_by("parcel_id", "date")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .sort("n", descending=True)
)
dup_extra_rows = dup_keys.select((pl.col("n") - 1).alias("extra")).to_series().sum() or 0
print(f"Duplicate (parcel_id, date) keys: {dup_keys.height} keys, {dup_extra_rows} extra rows")
display(dup_keys)

if dup_extra_rows:
    issue_rows.append(
        build_issue_row(
            "readings",
            "date",
            "Duplicate parcel_id + date combinations",
            int(dup_extra_rows),
            N_READINGS,
            "Drop + Flag",
            "Deduplicate by keeping the first occurrence or field with proper all fields are valid; log/flag the incident to trace upstream ingestion bugs.",
        )
    )

print("\n" + "=" * 60)
print("ndvi_value")
print("=" * 60)

ndvi_f = readings.with_columns(
    pl.col("ndvi_value").cast(pl.Float64, strict=False).alias("ndvi_f")
)
ndvi_cast_fail = ndvi_f.filter(pl.col("ndvi_f").is_null() & ~empty_or_null_mask("ndvi_value")).height
ndvi_below = ndvi_f.filter(pl.col("ndvi_f") < NDVI_MIN).height
ndvi_above = ndvi_f.filter(pl.col("ndvi_f") > NDVI_MAX).height
ndvi_out_of_range = ndvi_below + ndvi_above

print(f"NDVI cast failures: {ndvi_cast_fail} ({pct(ndvi_cast_fail, N_READINGS)})")
print(f"NDVI below {NDVI_MIN}: {ndvi_below} ({pct(ndvi_below, N_READINGS)})")
print(f"NDVI above {NDVI_MAX}: {ndvi_above} ({pct(ndvi_above, N_READINGS)})")
print(f"NDVI out of range total: {ndvi_out_of_range} ({pct(ndvi_out_of_range, N_READINGS)})")

if ndvi_out_of_range:
    invalid_stats = ndvi_f.filter(
        (pl.col("ndvi_f") < NDVI_MIN) | (pl.col("ndvi_f") > NDVI_MAX)
    ).select(
        pl.col("ndvi_f").min().alias("min_ndvi"),
        pl.col("ndvi_f").max().alias("max_ndvi"),
    )
    display(invalid_stats)
    issue_rows.append(
        build_issue_row(
            "readings",
            "ndvi_value",
            f"NDVI outside valid range [{NDVI_MIN}, {NDVI_MAX}]",
            ndvi_out_of_range,
            N_READINGS,
            "Flag + Nullify",
            "Physically impossible telemetry. Flag for data lineage tracking, coerce invalid values to NaN to avoid skewing ML models.",
        )
    )

print("\n" + "=" * 60)
print("temperature_c / rainfall_mm")
print("=" * 60)

for col in ["temperature_c", "rainfall_mm"]:
    num = readings.with_columns(
        pl.col(col).cast(pl.Float64, strict=False).alias("val")
    )
    cast_fail = num.filter(pl.col("val").is_null() & ~empty_or_null_mask(col)).height
    whole = num.filter(pl.col("val").is_not_null() & (pl.col("val") % 1 == 0)).height
    fractional = num.filter(pl.col("val").is_not_null() & (pl.col("val") % 1 != 0)).height
    print(f"\n{col}:")
    print(f"  Cast failures: {cast_fail} ({pct(cast_fail, N_READINGS)})")
    print(f"  Whole-number values: {whole} ({pct(whole, N_READINGS)})")
    print(f"  Fractional values: {fractional} ({pct(fractional, N_READINGS)})")

issue_rows.append(
    build_issue_row(
        "readings",
        "temperature_c",
        "Mixed int/float numeric representation (whole vs fractional)",
        readings.with_columns(pl.col("temperature_c").cast(pl.Float64).alias("v"))
        .filter(pl.col("v") % 1 == 0)
        .height,
        N_READINGS,
        "repair",
        "Cast to Float64 for consistent numeric operations in pipeline.",
    )
)
issue_rows.append(
    build_issue_row(
        "readings",
        "rainfall_mm",
        "Mixed int/float numeric representation (whole vs fractional)",
        readings.with_columns(pl.col("rainfall_mm").cast(pl.Float64).alias("v"))
        .filter(pl.col("v") % 1 == 0)
        .height,
        N_READINGS,
        "repair",
        "Cast to Float64 for consistent numeric operations in pipeline.",
    )
)

print("\n" + "=" * 60)
print("sensor_status")
print("=" * 60)

status_raw = (
    readings.group_by("sensor_status")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print("Raw value counts:")
display(status_raw)

ws_status = has_leading_trailing_ws(readings, "sensor_status").height
print(f"Rows with leading/trailing whitespace: {ws_status} ({pct(ws_status, N_READINGS)})")

if ws_status:
    issue_rows.append(
        build_issue_row(
            "readings",
            "sensor_status",
            "Leading/trailing whitespace in sensor_status",
            ws_status,
            N_READINGS,
            "repair",
            "Strip whitespace before normalizing status values.",
        )
    )

status_norm = (
    readings.with_columns(normalize_sensor_status())
    .group_by("sensor_status_normalized")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)
print("\nNormalized value counts (strip + uppercase, NA/empty -> MISSING):")
display(status_norm)

missing_status = (
    readings.with_columns(normalize_sensor_status())
    .filter(pl.col("sensor_status_normalized") == "MISSING")
    .height
)
print(f"Missing/NA/empty sensor_status: {missing_status} ({pct(missing_status, N_READINGS)})")

if missing_status:
    issue_rows.append(
        build_issue_row(
            "readings",
            "sensor_status",
            "Empty, NA, or NaN sensor_status",
            missing_status,
            N_READINGS,
            "flag",
            "Missing sensor health; exclude from analysis where status matters.",
        )
    )

variant_count = status_raw.height
if variant_count > 3:
    issue_rows.append(
        build_issue_row(
            "readings",
            "sensor_status",
            "Inconsistent status casing/spelling (OK/error/ERROR/ok etc.)",
            N_READINGS - missing_status,
            N_READINGS,
            "repair",
            "Normalize to canonical OK / ERROR / Missing after stripping whitespace.",
        )
    )

parcel_id
Null/empty parcel_id: 0 (0.0%)
Whitespace in parcel_id: 0 (0.0%)
Orphan parcel_ids (not in metadata): 2 parcels, 40 rows


parcel_id
str
"""PARCEL_098"""
"""PARCEL_099"""



date


date_format,count
str,u32
"""ISO (YYYY-MM-DD)""",2431
"""DD/MM/YYYY""",686
"""DD-Mon-YYYY""",330


Unparseable dates: 0 (0.0%)
Ambiguous slash dates (day<=12 and month<=12): 60 unique dates, 279 rows
Duplicate (parcel_id, date) keys: 8 keys, 8 extra rows


...


temperature_c / rainfall_mm

temperature_c:
  Cast failures: 0 (0.0%)
  Whole-number values: 349 (10.1%)
  Fractional values: 3098 (89.9%)

rainfall_mm:
  Cast failures: 0 (0.0%)
  Whole-number values: 2833 (82.2%)
  Fractional values: 614 (17.8%)

sensor_status
Raw value counts:


sensor_status,count
str,u32
"""OK""",2890
"""Error""",90
"""ERROR""",80
"""error""",76
"""ok""",64
"""OK """,57
""" OK""",53
"""NA""",53
null,43


Rows with leading/trailing whitespace: 110 (3.2%)

Normalized value counts (strip + uppercase, NA/empty -> MISSING):


sensor_status_normalized,count
str,u32
"""OK""",3064
"""ERROR""",246
"""MISSING""",94
null,43


Missing/NA/empty sensor_status: 94 (2.7%)


## parcel_metadata.csv audit

In [6]:
print("=" * 60)
print("Null / empty checks")
print("=" * 60)

for col in META_COLS:
    n = metadata.filter(empty_or_null_mask(col)).height
    print(f"{col}: {n} null/empty ({pct(n, N_METADATA)})")

print("\n" + "=" * 60)
print("Hidden whitespace checks")
print("=" * 60)

for col in META_COLS:
    n = has_leading_trailing_ws(metadata, col).height
    print(f"{col}: {n} rows with whitespace ({pct(n, N_METADATA)})")

print("\n" + "=" * 60)
print("parcel_id uniqueness")
print("=" * 60)

dup_meta = (
    metadata.group_by("parcel_id")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
)
print(f"Duplicate parcel_id: {dup_meta.height}")

print("\n" + "=" * 60)
print("area_hectares")
print("=" * 60)

area = metadata.with_columns(
    pl.col("area_hectares").cast(pl.Float64, strict=False).alias("area_f")
)
area_fail = area.filter(
    pl.col("area_f").is_null() & ~empty_or_null_mask("area_hectares")
).height
area_stats = area.select(
    pl.col("area_f").min().alias("min"),
    pl.col("area_f").max().alias("max"),
    pl.col("area_f").mean().alias("mean"),
)
print(f"Cast failures: {area_fail}")
display(area_stats)

print("\n" + "=" * 60)
print("sowing_date")
print("=" * 60)

sowing = metadata.with_columns(
    pl.col("sowing_date").str.to_date("%Y-%m-%d", strict=False).alias("sowing_parsed")
)
sowing_fail = sowing.filter(
    pl.col("sowing_parsed").is_null() & ~empty_or_null_mask("sowing_date")
).height
print(f"ISO parse failures: {sowing_fail} ({pct(sowing_fail, N_METADATA)})")

print("\n" + "=" * 60)
print("crop_type / mill_id distributions")
print("=" * 60)

display(metadata.group_by("crop_type").agg(pl.len().alias("count")).sort("count", descending=True))
display(metadata.group_by("mill_id").agg(pl.len().alias("count")).sort("count", descending=True))

print("\n" + "=" * 60)
print("Metadata parcels with no readings")
print("=" * 60)

reading_ids = readings.select("parcel_id").unique()
no_readings = (
    metadata.select("parcel_id")
    .join(reading_ids, on="parcel_id", how="anti")
    .sort("parcel_id")
)
print(f"Parcels in metadata but not in readings: {no_readings.height}")
display(no_readings)

if no_readings.height:
    issue_rows.append(
        build_issue_row(
            "metadata",
            "parcel_id",
            "Metadata parcels with zero readings",
            no_readings.height,
            N_METADATA,
            "flag",
            "Expected operational variance (e.g., new parcels). Flag to monitor pipeline coverage without dropping metadata.",
        )
    )

Null / empty checks
parcel_id: 0 null/empty (0.0%)
mill_id: 0 null/empty (0.0%)
crop_type: 0 null/empty (0.0%)
sowing_date: 0 null/empty (0.0%)
area_hectares: 0 null/empty (0.0%)

Hidden whitespace checks
parcel_id: 0 rows with whitespace (0.0%)
mill_id: 0 rows with whitespace (0.0%)
crop_type: 0 rows with whitespace (0.0%)
sowing_date: 0 rows with whitespace (0.0%)
area_hectares: 0 rows with whitespace (0.0%)

parcel_id uniqueness
Duplicate parcel_id: 0

area_hectares
Cast failures: 0


min,max,mean
f64,f64,f64
1.55,10.19,5.787143



sowing_date
ISO parse failures: 0 (0.0%)

crop_type / mill_id distributions


crop_type,count
str,u32
"""sugarcane""",20
"""soybean""",6
"""wheat""",2


mill_id,count
str,u32
"""MILL_NORTH""",9
"""MILL_EAST""",9
"""MILL_SOUTH""",5
"""MILL_WEST""",5



Metadata parcels with no readings
Parcels in metadata but not in readings: 3


parcel_id
str
"""PARCEL_050"""
"""PARCEL_051"""
"""PARCEL_052"""


## Cross-file integrity

In [12]:
print("=" * 60)
print("Readings left join metadata")
print("=" * 60)

joined = readings.join(metadata, on="parcel_id", how="left")
unmatched_readings = joined.filter(pl.col("mill_id").is_null()).height
print(f"Reading rows without metadata match: {unmatched_readings} ({pct(unmatched_readings, N_READINGS)})")
display(joined.filter(pl.col("mill_id").is_null()))

print("\n" + "=" * 60)
print("Metadata parcels with no readings")
print("=" * 60)

meta_missing = metadata.join(readings.select("parcel_id").unique(), on="parcel_id", how="anti")
print(f"Metadata parcels with no readings: {meta_missing.height}")
display(meta_missing)

print("\n" + "=" * 60)
print("Reading date range sanity")
print("=" * 60)

date_range = (
    readings.with_columns(parse_dates_multi("date"))
    .filter(pl.col("parsed_date").is_not_null())
    .select(
        pl.col("parsed_date").min().alias("min_date"),
        pl.col("parsed_date").max().alias("max_date"),
    )
)
display(date_range)

out_of_window = (
    readings.with_columns(parse_dates_multi("date"))
    .filter(
        pl.col("parsed_date").is_not_null()
        & (
            (pl.col("parsed_date") < pl.date(2026, 1, 1))
            | (pl.col("parsed_date") > pl.date(2026, 5, 31))
        )
    )
    .height
)
print(f"Readings outside Jan–May 2026: {out_of_window} ({pct(out_of_window, N_READINGS)})")

if out_of_window:
    issue_rows.append(
        build_issue_row(
            "cross_file",
            "date",
            "Reading dates outside expected Jan–May 2026 window",
            out_of_window,
            N_READINGS,
            "flag",
            "Unexpected date range; review before time-series analysis.",
        )
    )

Readings left join metadata
Reading rows without metadata match: 40 (1.2%)


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares
str,str,str,str,str,str,str,str,str,str
"""PARCEL_099""","""19-Mar-2026""","""0.577""","""15.6""","""0.0""","""OK""",null,null,null,null
"""PARCEL_098""","""2026-01-02""","""0.038""","""19.6""","""4.1""","""ok""",null,null,null,null
"""PARCEL_099""","""2026-03-26""","""0.654""","""32.9""","""1.7""","""OK""",null,null,null,null
"""PARCEL_098""","""27/01/2026""","""0.181""","""18.9""","""6.0""","""OK""",null,null,null,null
"""PARCEL_098""","""2026-04-06""","""0.441""","""21.2""","""4.4""","""OK""",null,null,null,null
…,…,…,…,…,…,…,…,…,…
"""PARCEL_098""","""2026-04-05""","""0.555""","""19.9""","""0.0""","""OK""",null,null,null,null
"""PARCEL_098""","""2026-02-04""","""0.319""","""30.3""","""5.3""","""ok""",null,null,null,null
"""PARCEL_098""","""19/03/2026""","""0.633""","""28.6""","""1.7""","""OK""",null,null,null,null



Metadata parcels with no readings
Metadata parcels with no readings: 3


parcel_id,mill_id,crop_type,sowing_date,area_hectares
str,str,str,str,str
"""PARCEL_050""","""MILL_SOUTH""","""soybean""","""2026-02-01""","""1.55"""
"""PARCEL_051""","""MILL_NORTH""","""sugarcane""","""2026-01-10""","""7.72"""
"""PARCEL_052""","""MILL_WEST""","""soybean""","""2026-01-05""","""4.92"""



Reading date range sanity


min_date,max_date
date,date
2026-01-01,2026-05-31


Readings outside Jan–May 2026: 0 (0.0%)


## Summary: all data quality issues

In [8]:
audit_summary = (
    pl.DataFrame(issue_rows)
    .unique(subset=["dataset", "field", "issue"], keep="first")
    .sort(["dataset", "field", "count"], descending=[False, False, True])
)

print(f"Total distinct issues recorded: {audit_summary.height}")
with pl.Config(
    tbl_rows=-1,
    tbl_cols=-1,
    fmt_str_lengths=1000,
    tbl_width_chars=1000,
):
    display(audit_summary)


Total distinct issues recorded: 12


dataset,field,issue,count,pct,action,justification
str,str,str,i64,str,str,str
"""metadata""","""parcel_id""","""Metadata parcels with zero readings""",3,"""10.7%""","""flag""","""Expected operational variance (e.g., new parcels). Flag to monitor pipeline coverage without dropping metadata."""
"""readings""","""date""","""Non-ISO date format: DD/MM/YYYY""",686,"""19.9%""","""repair""","""Parse to canonical Date using explicit format list."""
"""readings""","""date""","""Non-ISO date format: DD-Mon-YYYY""",330,"""9.6%""","""repair""","""Parse to canonical Date using explicit format list."""
"""readings""","""date""","""Ambiguous DD/MM slash dates (both parts <= 12)""",279,"""8.1%""","""Repair + Verify""","""Safe parsing: Infer locale (DD/MM vs MM/DD) from unambiguous rows in the same batch before converting to ISO and set a flag is_data_ambiguous = 1."""
"""readings""","""date""","""Duplicate parcel_id + date combinations""",8,"""0.2%""","""Drop + Flag""","""Deduplicate by keeping the first occurrence or field with proper all fields are valid; log/flag the incident to trace upstream ingestion bugs."""
"""readings""","""ndvi_value""","""NDVI outside valid range [-1.0, 1.0]""",104,"""3.0%""","""Flag + Nullify""","""Physically impossible telemetry. Flag for data lineage tracking, coerce invalid values to NaN to avoid skewing ML models."""
"""readings""","""parcel_id""","""Orphan parcel_id not in metadata""",40,"""1.2%""","""flag""","""No metadata to join; exclude from joined output or alert upstream."""
"""readings""","""rainfall_mm""","""Mixed int/float numeric representation (whole vs fractional)""",2833,"""82.2%""","""repair""","""Cast to Float64 for consistent numeric operations in pipeline."""
"""readings""","""sensor_status""","""Inconsistent status casing/spelling (OK/error/ERROR/ok etc.)""",3353,"""97.3%""","""repair""","""Normalize to canonical OK / ERROR / Missing after stripping whitespace."""
